In [ ]:
import yfinance as yf
import pandas as pd
import os
import time

In [ ]:
# Define project root and path to data folder
project_root = os.path.abspath(os.path.join(os.getcwd(), ".."))
data_dir = os.path.join(project_root, "data", "raw")
os.makedirs(data_dir, exist_ok=True)

In [ ]:
tickers = ["AAPL", "MSFT", "NVDA", "AMZN", "JNJ", "JPM", "XOM", "CAT", "PG", "NEE"]
ticker_data = pd.DataFrame()


for ticker in tickers:
    data = yf.download(ticker, start='2020-01-01', end='2025-01-01', auto_adjust=True)
    #Only keep Adjusted Close and Volume
    features = data[['Close','Volume']].copy()
    features.columns = [f"{ticker}_Close", f"{ticker}_Volume"]
    ticker_data = pd.concat([ticker_data, features], axis=1)
    time.sleep(1)  #Avoiding API Limits

ticker_data.head()

In [ ]:
returns = ticker_data[[col for col in ticker_data.columns if "Close" in col]].pct_change().dropna()
volume = ticker_data[[col for col in ticker_data.columns if "Volume" in col]].iloc[1:]
ml_data = pd.concat([returns, volume], axis=1)
print(ml_data.head())

In [ ]:
path = os.path.join(data_dir,"ML_DATA.csv")
ml_data = pd.read_csv(path)

In [ ]:
train_data = pd.read_csv(path, parse_dates=["Date"])
train_data = train_data.sort_values("Date").reset_index(drop=True)
train_data

In [ ]:
tickers = ["AAPL", "MSFT", "NVDA", "AMZN", "JNJ", "JPM", "XOM", "CAT", "PG", "NEE"]
train_data["Market_Avg"] = train_data[[f"{t}_Close" for t in tickers]].mean(axis=1)


for t in tickers:
    r = train_data[f"{t}_Close"] 
    v = train_data[f"{t}_Volume"]
    
    #Lagged returns
    train_data[f"{t}_Lag1"] = r.shift(1)
    train_data[f"{t}_Lag5"] = r.shift(5)
    train_data[f"{t}_Lag10"] = r.shift(10)
    
    #Rolling volatility
    train_data[f"{t}_Volatility5"] = r.rolling(5).std()
    train_data[f"{t}_Volatility20"] = r.rolling(20).std()
    
    #Rolling mean
    train_data[f"{t}_Mean5"] = r.rolling(5).mean()
    train_data[f"{t}_Mean20"] = r.rolling(20).mean()
    
    #Rolling correlation with average of all stocks
    train_data[f"{t}_CrossCorr5"] = r.rolling(5).corr(train_data["Market_Avg"])
    
    #Volume trend
    train_data[f"{t}_Vol_Mean5"] = v.rolling(5).mean()
    train_data[f"{t}_Vol_Change"] = v.pct_change()

#Drop rows with NaN
df = train_data.dropna().reset_index(drop=True)


In [ ]:
path = os.path.join(data_dir,"TRAINING_DATA.csv")
df.to_csv(path)